<a href="https://colab.research.google.com/github/sadiashova/A-new-Bangladesh/blob/main/NLP_NEXUS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: Tesla T4


In [3]:
import subprocess
subprocess.run(["pip", "install", "-q", "datasets", "transformers", "torch", "emoji", "scikit-learn", "pandas", "numpy", "matplotlib", "accelerate", "sentencepiece"], check=False)
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU')

import os, json, random, re, time, warnings, unicodedata, shutil
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW, Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from datasets import load_dataset as hf_load_dataset

try:
    import emoji as emoji_lib
    EMOJI_OK = True
except ImportError:
    EMOJI_OK = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = {
    "dataset": "AridHasan/BanglaMultiHate",
    "seed": 42,
    "model_name": "csebuetnlp/banglabert",
    "max_seq_len": 128,
    "batch_size": 16,
    "lr": 2e-5,
    "num_epochs": 3,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "fp16": True,
    "embed_dim": 128,
    "hidden_dim": 256,
    "bilstm_dropout": 0.3,
    "bilstm_epochs": 10,
    "bilstm_lr": 1e-3,
    "bilstm_seeds": [42, 123, 7],
    "transformer_seeds": [42],
    "output_dir": "/content/results",
    "checkpoint_dir": "/content/checkpoints",
}
os.makedirs(CFG["output_dir"] + "/figures", exist_ok=True)
os.makedirs(CFG["checkpoint_dir"], exist_ok=True)

BANGLISH_MAP = {
    "khrap": "kharap", "kharab": "kharap", "vai": "bhai", "bai": "bhai",
    "bhaiya": "bhai", "sala": "shala", "haramee": "harami",
    "pagla": "pagol", "besya": "beshya",
    "ae": "e", "aa": "a", "ii": "i", "uu": "u",
}

def preprocess(text):
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"(.)\1{3,}", r"\1\1\1", text)
    if EMOJI_OK:
        text = emoji_lib.demojize(text, delimiters=(" ", " "))
    return " ".join(BANGLISH_MAP.get(w.lower(), w) for w in text.split()).strip()

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

# Load BanglaMultiHate with exact same 70/15/15 stratified split
ds = hf_load_dataset(CFG["dataset"])
raw = pd.concat([ds[s].to_pandas() for s in ds.keys()], ignore_index=True)
raw = raw.rename(columns={"comment": "text", "type_of_hate": "hate_type", "severity_of_hate": "severity", "target_of_hate": "target"})
hate_vals = raw["hate_type"].astype(str).str.strip().str.lower()
raw["hate_binary"] = (~hate_vals.isin(["non-hate", "none", "not hate", "nan", ""])).astype(int)
for col in ["hate_type", "severity", "target"]:
    raw[f"{col}_enc"] = pd.Categorical(raw[col].astype(str)).codes
raw = raw[raw["text"].astype(str).str.strip().astype(bool)].reset_index(drop=True)
raw["text"] = raw["text"].apply(preprocess)

train_df, temp = train_test_split(raw, test_size=0.30, random_state=CFG["seed"], stratify=raw["hate_binary"])
val_df, test_df = train_test_split(temp, test_size=0.50, random_state=CFG["seed"], stratify=temp["hate_binary"])
train_df, val_df, test_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

print(f"Loaded BanglaMultiHate: Train={len(train_df):,}, Val={len(val_df):,}, Test={len(test_df):,}")


def compute_metrics(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    labels = sorted(set(y_true) | set(y_pred))
    return {
        "accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "precision": round(precision_score(y_true, y_pred, average="macro", zero_division=0, labels=labels), 4),
        "recall":    round(recall_score(y_true, y_pred, average="macro", zero_division=0, labels=labels), 4),
        "macro_f1":  round(f1_score(y_true, y_pred, average="macro", zero_division=0, labels=labels), 4),
    }

def agg_metrics(ml):
    keys = ml[0].keys()
    return {k: round(float(np.mean([m[k] for m in ml])), 4) for k in keys} | \
           {f"{k}_std": round(float(np.std([m[k] for m in ml])), 4) for k in keys}

def plot_cm(y_true, y_pred, title, path):
    cm = confusion_matrix(y_true, y_pred)
    n = cm.shape[0]
    fig, ax = plt.subplots(figsize=(max(5, n*1.1), max(4, n*1.0)))
    ax.imshow(cm, cmap=plt.cm.Blues)
    ax.set(xticks=range(n), yticks=range(n), title=title, xlabel="Predicted", ylabel="True")
    thresh = cm.max() / 2
    for i in range(n):
        for j in range(n):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def _cw(labels, ncls):
    cnt = Counter(labels); tot = sum(cnt.values())
    return torch.tensor([tot / (ncls * cnt.get(i, 1)) for i in range(ncls)], dtype=torch.float).to(DEVICE)

# ── BiLSTM ────────────────────────────────────────────────────────────────────
class Vocab:
    def __init__(self, min_freq=2):
        self.w2i = {"<PAD>": 0, "<UNK>": 1}; self.min_freq = min_freq
    def build(self, texts):
        cnt = Counter(w for t in texts for w in t.split())
        for w, f in cnt.items():
            if f >= self.min_freq and w not in self.w2i:
                self.w2i[w] = len(self.w2i)
    def enc(self, text, maxlen=128):
        ids = [self.w2i.get(w, 1) for w in text.split()][:maxlen]
        return ids + [0] * (maxlen - len(ids))
    def __len__(self): return len(self.w2i)

class TextDS(Dataset):
    def __init__(self, texts, labels, vocab, maxlen=128):
        self.X = [torch.tensor(vocab.enc(t, maxlen), dtype=torch.long) for t in texts]
        self.y = [torch.tensor(l, dtype=torch.long) for l in labels]
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class BiLSTM(nn.Module):
    def __init__(self, vsz, edim, hdim, ncls, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding(vsz, edim, padding_idx=0)
        self.lstm = nn.LSTM(edim, hdim, num_layers=2, bidirectional=True, batch_first=True, dropout=drop)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(hdim * 2, ncls)
    def forward(self, x):
        e = self.drop(self.emb(x))
        _, (hn, _) = self.lstm(e)
        return self.fc(self.drop(torch.cat([hn[-2], hn[-1]], dim=-1)))

def run_bilstm(train_df, val_df, test_df, label_col="hate_binary"):
    X_tr, y_tr = train_df["text"].tolist(), train_df[label_col].tolist()
    X_v,  y_v  = val_df["text"].tolist(),   val_df[label_col].tolist()
    X_te, y_te = test_df["text"].tolist(),  test_df[label_col].tolist()
    ncls = len(set(y_tr))
    cw = _cw(y_tr, ncls)
    sv_m, tm_l, last_preds = [], [], None

    for seed in CFG["bilstm_seeds"]:
        set_seed(seed); print(f"\n── BiLSTM seed={seed} ──")
        vocab = Vocab(min_freq=2); vocab.build(X_tr)
        trl = DataLoader(TextDS(X_tr, y_tr, vocab), batch_size=64, shuffle=True)
        vl  = DataLoader(TextDS(X_v,  y_v,  vocab), batch_size=128)
        tel = DataLoader(TextDS(X_te, y_te, vocab), batch_size=128)
        model = BiLSTM(len(vocab), CFG["embed_dim"], CFG["hidden_dim"], ncls, CFG["bilstm_dropout"]).to(DEVICE)
        opt = Adam(model.parameters(), lr=CFG["bilstm_lr"])
        sch = ReduceLROnPlateau(opt, patience=2, factor=0.5)
        crit = nn.CrossEntropyLoss(weight=cw)
        best_f1, best_st = -1, None
        for ep in range(CFG["bilstm_epochs"]):
            model.train(); tloss = 0
            for xb, yb in trl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad(); loss = crit(model(xb), yb); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); tloss += loss.item()
            model.eval(); vp = []
            with torch.no_grad():
                for xb, _ in vl: vp += model(xb.to(DEVICE)).argmax(-1).cpu().tolist()
            vf1 = f1_score(y_v, vp, average="macro", zero_division=0)
            sch.step(1 - vf1)
            if vf1 > best_f1: best_f1 = vf1; best_st = {k: v.clone() for k, v in model.state_dict().items()}
            if (ep + 1) % 2 == 0:
                print(f"  Ep {ep+1}/{CFG['bilstm_epochs']} loss={tloss/len(trl):.4f} val_f1={vf1:.4f}")
        model.load_state_dict(best_st); model.eval()
        def _pred(loader):
            p = []
            with torch.no_grad():
                for xb, _ in loader: p += model(xb.to(DEVICE)).argmax(-1).cpu().tolist()
            return p
        sv_m.append(compute_metrics(y_v, _pred(vl)))
        last_preds = _pred(tel)
        tm_l.append(compute_metrics(y_te, last_preds))

    res = {"val": agg_metrics(sv_m), "test": agg_metrics(tm_l), "test_predictions": last_preds, "true_labels": y_te}
    print("BiLSTM Test:", res["test"])
    plot_cm(y_te, last_preds, "BiLSTM Confusion Matrix", f"{CFG['output_dir']}/figures/bilstm_cm.png")
    return res

# ── BanglaBERT Single-Task & Multi-Task ───────────────────────────────────────
class SingleTaskDS(Dataset):
    def __init__(self, texts, labels, tok, maxlen):
        self.enc = tok(texts, max_length=maxlen, padding="max_length", truncation=True, return_tensors="pt")
        self.lbl = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.lbl)
    def __getitem__(self, i): return {k: v[i] for k, v in self.enc.items()} | {"labels": self.lbl[i]}

class MultiTaskDS(Dataset):
    def __init__(self, texts, labels_dict, tok, maxlen):
        self.enc = tok(texts, max_length=maxlen, padding="max_length", truncation=True, return_tensors="pt")
        self.lbl = {k: torch.tensor(v, dtype=torch.long) for k, v in labels_dict.items()}
    def __len__(self): return len(next(iter(self.lbl.values())))
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        for t, v in self.lbl.items(): item[f"lbl_{t}"] = v[i]
        return item

class MTBanglaBERT(nn.Module):
    def __init__(self, model_name, tasks_cfg, drop=0.1):
        super().__init__()
        self.enc = AutoModel.from_pretrained(model_name)
        hsz = self.enc.config.hidden_size
        self.drop = nn.Dropout(drop)
        self.heads = nn.ModuleDict({t: nn.Linear(hsz, n) for t, n in tasks_cfg.items()})
    def forward(self, input_ids, attention_mask):
        out = self.enc(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.drop(out.last_hidden_state[:, 0, :])
        return {t: h(pooled) for t, h in self.heads.items()}

def run_single_task(train_df, val_df, test_df, label_col):
    tok = AutoTokenizer.from_pretrained(CFG["model_name"])
    X_tr, y_tr = train_df["text"].tolist(), train_df[label_col].tolist()
    X_v,  y_v  = val_df["text"].tolist(),   val_df[label_col].tolist()
    X_te, y_te = test_df["text"].tolist(),  test_df[label_col].tolist()
    ncls = len(set(y_tr))
    set_seed(42)
    print(f"\n── BanglaBERT SingleTask: {label_col} ({ncls} classes) ──")
    model = AutoModelForSequenceClassification.from_pretrained(CFG["model_name"], num_labels=ncls).to(DEVICE)
    trl = DataLoader(SingleTaskDS(X_tr, y_tr, tok, CFG["max_seq_len"]), batch_size=CFG["batch_size"], shuffle=True)
    vl  = DataLoader(SingleTaskDS(X_v,  y_v,  tok, CFG["max_seq_len"]), batch_size=CFG["batch_size"]*2)
    tel = DataLoader(SingleTaskDS(X_te, y_te, tok, CFG["max_seq_len"]), batch_size=CFG["batch_size"]*2)
    crit = nn.CrossEntropyLoss(weight=_cw(y_tr, ncls))
    opt = AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    total = len(trl) * CFG["num_epochs"]
    sch = get_linear_schedule_with_warmup(opt, int(total * CFG["warmup_ratio"]), total)
    scaler = torch.cuda.amp.GradScaler() if (CFG["fp16"] and DEVICE.type == "cuda") else None
    best_f1, best_st = -1, None

    for ep in range(CFG["num_epochs"]):
        model.train(); tloss = 0
        for b in trl:
            ids, mask, lbl = b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE), b["labels"].to(DEVICE)
            opt.zero_grad()
            if scaler:
                with torch.cuda.amp.autocast():
                    loss = crit(model(ids, attention_mask=mask).logits, lbl)
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
            else:
                loss = crit(model(ids, attention_mask=mask).logits, lbl)
                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            sch.step(); tloss += loss.item()
        model.eval(); vp = []
        with torch.no_grad():
            for b in vl:
                vp += model(b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE)).logits.argmax(-1).cpu().tolist()
        vf1 = f1_score(y_v, vp, average="macro", zero_division=0)
        print(f"  Ep {ep+1}/{CFG['num_epochs']} loss={tloss/len(trl):.4f} val_f1={vf1:.4f}")
        if vf1 > best_f1: best_f1 = vf1; best_st = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_st); model.eval()
    def _pred(loader):
        p = []
        with torch.no_grad():
            for b in loader:
                p += model(b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE)).logits.argmax(-1).cpu().tolist()
        return p
    preds = _pred(tel)
    res = {"val": compute_metrics(y_v, _pred(vl)), "test": compute_metrics(y_te, preds), "test_predictions": preds, "true_labels": y_te}
    print(f"SingleTask-{label_col} Test:", res["test"])
    plot_cm(y_te, preds, f"BanglaBERT ST – {label_col}", f"{CFG['output_dir']}/figures/banglabert_st_{label_col}_cm.png")
    return res

def run_multi_task(train_df, val_df, test_df, task_cols):
    tok = AutoTokenizer.from_pretrained(CFG["model_name"])
    tasks_cfg = {t: int(train_df[col].max()) + 1 for t, col in task_cols.items()}
    set_seed(42)
    print(f"\n── BanglaBERT MultiTask: {list(task_cols.keys())} ──")
    mk = lambda df: {t: df[col].tolist() for t, col in task_cols.items()}
    trl = DataLoader(MultiTaskDS(train_df["text"].tolist(), mk(train_df), tok, CFG["max_seq_len"]), batch_size=CFG["batch_size"], shuffle=True)
    vl  = DataLoader(MultiTaskDS(val_df["text"].tolist(),   mk(val_df),   tok, CFG["max_seq_len"]), batch_size=CFG["batch_size"]*2)
    tel = DataLoader(MultiTaskDS(test_df["text"].tolist(),  mk(test_df),  tok, CFG["max_seq_len"]), batch_size=CFG["batch_size"]*2)

    model = MTBanglaBERT(CFG["model_name"], tasks_cfg).to(DEVICE)
    crits = {t: nn.CrossEntropyLoss(weight=_cw(train_df[col].tolist(), tasks_cfg[t])) for t, col in task_cols.items()}
    opt = AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    total = len(trl) * CFG["num_epochs"]
    sch = get_linear_schedule_with_warmup(opt, int(total * CFG["warmup_ratio"]), total)
    scaler = torch.cuda.amp.GradScaler() if (CFG["fp16"] and DEVICE.type == "cuda") else None
    best_avg, best_st = -1, None

    for ep in range(CFG["num_epochs"]):
        model.train(); tloss = 0
        for b in trl:
            ids, mask = b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE)
            opt.zero_grad()
            if scaler:
                with torch.cuda.amp.autocast():
                    logd = model(ids, mask)
                    loss = sum(crits[t](logd[t], b[f"lbl_{t}"].to(DEVICE)) for t in task_cols)
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
            else:
                logd = model(ids, mask)
                loss = sum(crits[t](logd[t], b[f"lbl_{t}"].to(DEVICE)) for t in task_cols)
                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            sch.step(); tloss += loss.item()
        model.eval(); vp = {t: [] for t in task_cols}
        with torch.no_grad():
            for b in vl:
                ld = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
                for t in task_cols: vp[t] += ld[t].argmax(-1).cpu().tolist()
        avg_f1 = np.mean([f1_score(val_df[col].tolist(), vp[t], average="macro", zero_division=0) for t, col in task_cols.items()])
        print(f"  Ep {ep+1}/{CFG['num_epochs']} loss={tloss/len(trl):.4f} avg_val_f1={avg_f1:.4f}")
        if avg_f1 > best_avg: best_avg = avg_f1; best_st = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_st); model.eval()
    def _pred_mt(loader):
        pd_ = {t: [] for t in task_cols}
        with torch.no_grad():
            for b in loader:
                ld = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
                for t in task_cols: pd_[t] += ld[t].argmax(-1).cpu().tolist()
        return pd_
    vp_fin, tep_fin = _pred_mt(vl), _pred_mt(tel)
    results = {}
    for t, col in task_cols.items():
        results[t] = {
            "val": compute_metrics(val_df[col].tolist(), vp_fin[t]),
            "test": compute_metrics(test_df[col].tolist(), tep_fin[t]),
            "test_predictions": tep_fin[t],
            "true_labels": test_df[col].tolist(),
        }
        print(f"MultiTask-{t} Test:", results[t]["test"])
        plot_cm(test_df[col].tolist(), tep_fin[t], f"BanglaBERT MT – {t}", f"{CFG['output_dir']}/figures/banglabert_mt_{t}_cm.png")
    return results


t0 = time.time()
ALL_GPU_RESULTS = {}

# 1. BiLSTM (binary hate)
ALL_GPU_RESULTS["BiLSTM_hate_binary"] = run_bilstm(train_df, val_df, test_df, "hate_binary")

# 2. BanglaBERT Single-Task (binary + each multi-task label)
ALL_GPU_RESULTS["BanglaBERT_ST_hate_binary"] = run_single_task(train_df, val_df, test_df, "hate_binary")

TASK_COLS = {"hate_type": "hate_type_enc", "severity": "severity_enc", "target": "target_enc"}
for t, col in TASK_COLS.items():
    ALL_GPU_RESULTS[f"BanglaBERT_ST_{t}"] = run_single_task(train_df, val_df, test_df, col)

# 3. BanglaBERT Multi-Task (RQ1)
mt_res = run_multi_task(train_df, val_df, test_df, TASK_COLS)
for t, r in mt_res.items():
    ALL_GPU_RESULTS[f"BanglaBERT_MT_{t}"] = r

# 4. RQ1 Table: Multi-task vs Single-task
rq1_rows = []
for t in TASK_COLS:
    st_f1 = ALL_GPU_RESULTS[f"BanglaBERT_ST_{t}"]["test"]["macro_f1"]
    mt_f1 = ALL_GPU_RESULTS[f"BanglaBERT_MT_{t}"]["test"]["macro_f1"]
    rq1_rows.append({"Task": t, "SingleTask_F1": st_f1, "MultiTask_F1": mt_f1, "Delta": round(mt_f1 - st_f1, 4)})
rq1_df = pd.DataFrame(rq1_rows)
print("\n=== RQ1: Multi-Task vs Single-Task BanglaBERT ===")
print(rq1_df.to_string(index=False))
rq1_df.to_csv(f"{CFG['output_dir']}/rq1_mt_vs_st.csv", index=False)

# 5. Summary Table
rows = []
for name, res in ALL_GPU_RESULTS.items():
    m = res["test"]
    rows.append({"Model": name, "Accuracy": m["accuracy"], "Precision": m["precision"], "Recall": m["recall"], "Macro_F1": m["macro_f1"]})
summary_df = pd.DataFrame(rows)
print("\n=== GPU Models Summary (Test Set) ===")
print(summary_df.to_string(index=False))
summary_df.to_csv(f"{CFG['output_dir']}/gpu_results_table.csv", index=False)

# Save JSON
out_json = {m: {"val": r["val"], "test": r["test"]} for m, r in ALL_GPU_RESULTS.items()}
with open(f"{CFG['output_dir']}/gpu_all_results.json", "w") as f:
    json.dump(out_json, f, indent=2)

# Zip everything for 1-click download
shutil.make_archive("/content/results", "zip", CFG["output_dir"])
print(f"\n✓ Done in {(time.time()-t0)/60:.1f} min! Download /content/results.zip from the left Files panel.")


PyTorch: 2.14.1+cu130 | CUDA: True
GPU: Tesla T4
Loaded BanglaMultiHate: Train=35,522, Val=7,612, Test=7,612

── BiLSTM seed=42 ──
  Ep 2/10 loss=0.5455 val_f1=0.7291
  Ep 4/10 loss=0.4592 val_f1=0.7407
  Ep 6/10 loss=0.3716 val_f1=0.7413
  Ep 8/10 loss=0.2935 val_f1=0.7355
  Ep 10/10 loss=0.2064 val_f1=0.7243

── BiLSTM seed=123 ──
  Ep 2/10 loss=0.5444 val_f1=0.7282
  Ep 4/10 loss=0.4725 val_f1=0.7401
  Ep 6/10 loss=0.3765 val_f1=0.7517
  Ep 8/10 loss=0.2977 val_f1=0.7363
  Ep 10/10 loss=0.2123 val_f1=0.7352

── BiLSTM seed=7 ──
  Ep 2/10 loss=0.5489 val_f1=0.7318
  Ep 4/10 loss=0.4470 val_f1=0.7403
  Ep 6/10 loss=0.4017 val_f1=0.7392
  Ep 8/10 loss=0.2832 val_f1=0.7321
  Ep 10/10 loss=0.2290 val_f1=0.7323
BiLSTM Test: {'accuracy': 0.7468, 'precision': 0.7446, 'recall': 0.7383, 'macro_f1': 0.7397, 'accuracy_std': 0.0027, 'precision_std': 0.0007, 'recall_std': 0.0068, 'macro_f1_std': 0.0055}

── BanglaBERT SingleTask: hate_binary (2 classes) ──


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Ep 1/3 loss=0.4763 val_f1=0.7981
  Ep 2/3 loss=0.3575 val_f1=0.8021
  Ep 3/3 loss=0.2652 val_f1=0.7973
SingleTask-hate_binary Test: {'accuracy': 0.8043, 'precision': 0.8014, 'recall': 0.8052, 'macro_f1': 0.8025}

── BanglaBERT SingleTask: hate_type_enc (6 classes) ──


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (6 outputs left out)

  Ep 1/3 loss=1.1805 val_f1=0.5808
  Ep 2/3 loss=0.8720 val_f1=0.5670
  Ep 3/3 loss=0.7004 val_f1=0.5773
SingleTask-target_enc Test: {'accuracy': 0.6741, 'precision': 0.5553, 'recall': 0.6398, 'macro_f1': 0.5791}

── BanglaBERT MultiTask: ['hate_type', 'severity', 'target'] ──


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Ep 1/3 loss=3.2186 avg_val_f1=0.5580
  Ep 2/3 loss=2.3925 avg_val_f1=0.5839
  Ep 3/3 loss=2.0283 avg_val_f1=0.5984
MultiTask-hate_type Test: {'accuracy': 0.6888, 'precision': 0.5096, 'recall': 0.6309, 'macro_f1': 0.5518}
MultiTask-severity Test: {'accuracy': 0.7099, 'precision': 0.6288, 'recall': 0.6707, 'macro_f1': 0.6398}
MultiTask-target Test: {'accuracy': 0.6825, 'precision': 0.5535, 'recall': 0.6493, 'macro_f1': 0.5855}

=== RQ1: Multi-Task vs Single-Task BanglaBERT ===
     Task  SingleTask_F1  MultiTask_F1   Delta
hate_type         0.5675        0.5518 -0.0157
 severity         0.6298        0.6398  0.0100
   target         0.5791        0.5855  0.0064

=== GPU Models Summary (Test Set) ===
                    Model  Accuracy  Precision  Recall  Macro_F1
       BiLSTM_hate_binary    0.7468     0.7446  0.7383    0.7397
BanglaBERT_ST_hate_binary    0.8043     0.8014  0.8052    0.8025
  BanglaBERT_ST_hate_type    0.6992     0.5315  0.6229    0.5675
   BanglaBERT_ST_severity    0.